**Week 10**<br />Wednesday, Nov. 4

# Machine Learning, Continued

LX 394/694<br />
Coding for Linguists

Sophie Hao


## Announcements

- Due to an error, HW 2 submissions will be accepted without penalty until Monday, Nov. 9, at 9 AM.
- HW 3 will be released **next week** on schedule.
- We are now on Chapter 5 of the textbook (Tagging).
- This is the first "real" computational linguistics topic we are covering.
- BUCLD is this week!

## Recap: Tagging

**Tagging** is an NLP task in which each token in a text gets assigned a **tag.**

**Example:** Part of speech (POS) tagging

In [1]:
import nltk

text = nltk.word_tokenize("The quick brown fox jumps over the lazy dog.")
nltk.pos_tag(text)

[('The', 'DT'),
 ('quick', 'JJ'),
 ('brown', 'NN'),
 ('fox', 'NN'),
 ('jumps', 'VBZ'),
 ('over', 'IN'),
 ('the', 'DT'),
 ('lazy', 'JJ'),
 ('dog', 'NN'),
 ('.', '.')]

## Part of Speech Tagging

<br />

| The | quick | brown | fox | jumps | over | the | lazy | dog | . |
|-----|-------|-------|-----|-------|------|-----|------|-----|---|
| DT  | JJ    | NN    | NN  | VBZ   | IN   | DT  | JJ   | NN  | . |

<br />

| Tag | Description | Tag | Description |
|-----|-------------|-----|-------------|
| DT  | Determiner  | VBZ | Verb, 3rd person singular present |
| JJ  | Adjective   | IN  | Preposition or subordinating conjunction |
| NN  | Noun, singular or mass | . | Period |

[See full list of tags](https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html)

## Other Tagging Tasks


**Named Entity Recognition:**

| U.N. | official | Ekeus | heads | for | Baghdad | . |
|------|----------|-------|-------|-----|---------|---|
| ORG  |          | PER   |       |     | LOC     |   |

<span style="color: #cccccc">From <a href="https://aclanthology.org/W03-0419/" style="color: #cccccc">Tjong Kim Sang and De Meulder (2003)</a></span>


**Intent Detection/Slot Filling:**

| Set the temperature | to | 23°C | in | the | living room | . |
|---------------------|----|------|----|-----|-------------|---|
| intent              |    | temperature | | | room        |   |

<span style="color: #cccccc">From <a href="https://arxiv.org/abs/1805.10190" style="color: #cccccc">Coucke et al. (2018)</a></span>

## Recap: Unigram and Bigram Taggers

Last time, we explored **statistical** methods for tagging based on **tagged corpora**.

- **Unigram Tagger:** The tag for token $w$ is the most common tag for $w$ in the corpus
- **Bigram Tagger:** The tag for token $w$ followed by a token with tag $t$ is the most common tag for $w$ in the corpus, when it is followed by a token of tag $t$

## Mathematical Description

Let $w_1, w_2, \dots, w_n$ be a text, let $\hat{t}_i$ be the tagger's prediction for the tag of $w_i$, and let $t_i$ be actual tag for $w_i$ (i.e., the **ground-truth** or **gold-standard** tag).

Let $c$ be a **training corpus**.

- **Unigram Tagger:** $$\hat{t}_i = \text{argmax}_t \, \mathbb{P}[t \mathrel{|} w_i] = \text{argmax}_t \, \frac{c\texttt{.count}((w_i, t))}{\texttt{len}(c)}$$

- **Bigram Tagger:** $$\hat{t}_i = \text{argmax}_t \, \mathbb{P}[t \mathrel{|} t_{i - 1}w_i] = \text{argmax}_t \, \frac{\sum_w c\texttt{.count}((w, t_{i - 1})(w_i, t))}{\sum_w c\texttt{.count}((w, t_{i - 1}))}$$

## Mathematical Description

Let $w_1, w_2, \dots, w_n$ be a text, let $\hat{t}_i$ be the tagger's prediction for the tag of $w_i$, and let $t_i$ be actual tag for $w_i$ (i.e., the **ground-truth** or **gold-standard** tag).

Let $c$ be a **training corpus**.

- **Unigram Tagger:** $$\hat{t}_i = \text{argmax}_t \, c\texttt{.count}((w_i, t))$$
- **Bigram Tagger:** $$\hat{t}_i = \text{argmax}_t \, \sum_w c\texttt{.count}((w, t_{i - 1})(w_i, t))$$

## Training a Unigram Tagger

In [ ]:
import nltk

unigram = nltk.UnigramTagger(brown.tagged_sents())

In [ ]:
text = nltk.word_tokenize("The quick brown fox jumps over the lazy dog.")
unigram.tag(text)

## Bigram Tagger

A **bigram** tagger guesses the tag of a token based on the tag of the previous token.

In [ ]:
bigram = nltk.BigramTagger(brown.tagged_sents())
bigram.tag(text)

## Recap: Training and Evaluating Taggers

Which method of tagging is the best? We need to **evaluate** our taggers by comparing their performance on a **test set**.

In [2]:
import nltk
from nltk.corpus import brown

# Train–test split
dataset = brown.tagged_sents(categories="news")
n = int(len(dataset) * .8)
train_set = dataset[:n]
test_set = dataset[n:]

# Train
unigram = nltk.UnigramTagger(train_set)
bigram = nltk.BigramTagger(train_set)

## Accuracy

Let's measure the **accuracy** of each tagger on our test set: what percentage of tokens are tagged correctly?

$$\text{accuracy} = \frac{\sum_{(w_i, t_i) \in c} \texttt{int}(t_i = \hat{t}_i)}{\texttt{len}(c)}$$

In [3]:
# Test
print(f"Unigram Tagger:\t{unigram.accuracy(test_set):.3f}")
print(f"Bigram Tagger:\t{bigram.accuracy(test_set):.3f}")

Unigram Tagger:	0.803
Bigram Tagger:	0.092


## Error Analysis

The bigram tagger is surprisingly bad. Why?

In [5]:
from nltk.probability import FreqDist

# What is the most common incorrect tag?
error_fd = FreqDist()

for sent in test_set:
    words, tags = zip(*sent)
    _, pred_tags = zip(*bigram.tag(words))
    error_fd.update([pt for t, pt in zip(tags, pred_tags) if t != pt])
    
error_fd.most_common(5)

[(None, 18743), ('IN', 16), ('CS', 13), ('TO', 12), ('RB', 9)]

## Backoff Tagger

A **backoff tagger** uses a different tagger if it doesn't know how to tag a particular token.

In [7]:


regex = nltk.RegexpTagger(patterns)
unigram_backoff = nltk.UnigramTagger(train_set, backoff=regex)
bigram_backoff = nltk.BigramTagger(train_set, backoff=unigram_backoff)

In [9]:
# Evaluate
print(f"Regex Tagger:\t {regex.accuracy(test_set):.3f}")
print(f"Unigram Tagger:\t {unigram.accuracy(test_set):.3f}")
print(f"Bigram Tagger:\t {bigram.accuracy(test_set):.3f}\n")

print(f"Unigram Backoff: {unigram_backoff.accuracy(test_set):.3f}")
print(f"Bigram Backoff:\t {bigram_backoff.accuracy(test_set):.3f}\n")

Regex Tagger:	 0.196
Unigram Tagger:	 0.803
Bigram Tagger:	 0.092

Unigram Backoff: 0.849
Bigram Backoff:	 0.858



## Baselines

When we create our own machine learning model, we need to compare it to **baseline models**. We usually want our model to be better than the baseline models in some way.

Baseline models usually include:
- the worst possible way of doing the task
- the current most typical way of doing the task
- the best possible way of doing the task

## Example: Olympic Ski Halfpipe Qualification Round

**Elizabeth Swaney** is an American, Venezuelan, Hungarian, and Israeli skier who represented Hungary at the 2018 Winter Olympics. She happens to be Sophie's favorite athlete.

Swaney is not a professional-level skier, but qualified for the Olympics mainly by [exploiting loopholes](https://en.wikipedia.org/wiki/Elizabeth_Swaney#Olympic_qualifying). She is therefore a natural baseline for measuring the performance of professional-level Olympic skiers.

| Rank | Score | Athlete | NOC |
|---|---|---|---|
| 1 | **93.40** | Cassie Sharpe | 🇨🇦 |
| 2 | 92.00 | Marie Martinod | 🇫🇷 |
| 3 | 90.60 | Brita Sigourney | 🇺🇸 |
| 4 | 86.00 | Annalisa Drew | 🇺🇸 |
| 5 | 84.80 | Ayana Onozuka | 🇯🇵 |
| … | … | … | … |
| 24 | 31.40 | Elizabeth Swaney | 🇭🇺 |

<a href="https://en.wikipedia.org/wiki/Freestyle_skiing_at_the_2018_Winter_Olympics_%E2%80%93_Women's_halfpipe" style="color: #cccccc">Source: Wikipedia</a>

## Random Baseline

The **random baseline** is a model that just guesses its output randomly (with uniform probability).

Note that we don't need to actually create the random baseline model in order to calculate its accuracy.

In [11]:
# Count how many tags there are in the training data
tagset = set()
for sent in train_set:
    _, tags = zip(*sent)
    tagset.add(tags)
len(tagset)

3571

In [12]:
# Random baseline accuracy
1 / len(tagset)

0.0002800336040324839

## Next Time

Machine Learning, Continued